# Source-validation metrics
Written only; no cells executed. See task2/README.md for execution order.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
@torch.no_grad()
def evaluate(model, loaders, device):
    model.eval()
    rows, predictions = [], []
    for domain, loader in loaders.items():
        truth, pred, losses = [], [], 0.0
        for x, y in loader:
            logits = model(x.to(device))
            losses += nn.functional.cross_entropy(logits, y.to(device), reduction='sum').item()
            truth.extend(y.tolist()); pred.extend(logits.argmax(1).cpu().tolist())
        rows.append(dict(domain=domain, accuracy=accuracy_score(truth, pred),
                         macro_f1=f1_score(truth, pred, labels=list(range(7)), average='macro', zero_division=0),
                         loss=losses / len(truth), count=len(truth)))
        predictions.extend(dict(domain=domain, path=r['path'], label=y, prediction=p)
                           for r, y, p in zip(loader.dataset.records, truth, pred))
    return pd.DataFrame(rows), pd.DataFrame(predictions)


